# Evaluate Model with sampled MMLU datas

**Guide: Model **

In [ ]:
# directory setting

In [ ]:
import sys
sys.path.append(os.path.abspath(".."))

In [ ]:
from config import Cofig as cfg
import os
import json

Mounted at /content/drive


In [ ]:
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
from datasets import Dataset, load_dataset

In [ ]:
import torch
import random
import numpy as np

seed = cfg.SEED
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)

# Base model load

In [ ]:
#base model
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
model_name =cfg.BASE_MODEL_NAME
# 1. base model load
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# 2. tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

# Trained model Load

In [ ]:
#Trained model
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
model_name = cfg.TRAINED_MODEL_NAME
# 1. trained model load
tmodel = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# 2. tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
dataset =  load_dataset('cais/mmlu','all')

README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

all/dev-00000-of-00001.parquet:   0%|          | 0.00/76.5k [00:00<?, ?B/s]

all/auxiliary_train-00000-of-00001.parqu(…):   0%|          | 0.00/47.5M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/14042 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

In [ ]:
import torch
from collections import defaultdict

# ---------------------------
# 1. data grouping
# ---------------------------
def build_subject_mapping(dataset):
    subject2dev = defaultdict(list)
    subject2test = defaultdict(list)

    for x in dataset['dev']:
        subject2dev[x['subject']].append(x)

    for x in dataset['test']:
        subject2test[x['subject']].append(x)

    return subject2dev, subject2test


# ---------------------------
# 2. prompt generation
# ---------------------------
def format_example(x):
    text = f"Question: {x['question']}\n"
    for i, choice in enumerate(x['choices']):
        text += f"{chr(65+i)}. {choice}\n"
    text += f"Answer: {chr(65 + x['answer'])}\n\n"
    return text


def build_prompt(shots, sample):
    prompt = ""

    # few-shot examples
    for shot in shots:
        prompt += format_example(shot)

    # test question (no answer)
    prompt += f"Question: {sample['question']}\n"
    for i, choice in enumerate(sample['choices']):
        prompt += f"{chr(65+i)}. {choice}\n"
    prompt += "Answer:"

    return prompt


# ---------------------------
# 3. logit prediction
# ---------------------------
def predict(model, tokenizer, prompt, device):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    # position of last token
    last_logits = logits[0, -1]

    # choice token ids (" A", " B", ...)
    choices = ["A", "B", "C", "D"]
    choice_ids = [
        tokenizer.encode(" " + c, add_special_tokens=False)[-1]
        for c in choices
    ]

    # calculation probability
    probs = torch.softmax(last_logits, dim=-1)
    scores = [probs[i].item() for i in choice_ids]

    # choosing one
    pred = choices[scores.index(max(scores))]

    return pred


# ---------------------------
# 4. evaluation loop
# ---------------------------
def evaluate(model, tokenizer, dataset, device, max_samples_per_subject=None):
    subject2dev, subject2test = build_subject_mapping(dataset)

    results = {}

    for subject in subject2test:

        # 5-shot
        shots = subject2dev[subject][:5]

        correct = 0
        total = 0

        test_samples = subject2test[subject]

        #  limit sample
        if max_samples_per_subject:
            test_samples = test_samples[:max_samples_per_subject]

        for sample in test_samples:
            prompt = build_prompt(shots, sample)

            pred = predict(model, tokenizer, prompt, device)
            gt = chr(65 + sample["answer"])

            if pred == gt:
                correct += 1
            total += 1

        acc = correct / total
        results[subject] = acc

        print(f"{subject}: {acc:.4f}")

    # total average
    final_score = sum(results.values()) / len(results)

    print(f"\nFinal MMLU Score: {final_score:.4f}")

    return results, final_score

In [ ]:
results, score = evaluate(
    model,
    tokenizer,
    dataset,
    device=model.device,
    max_samples_per_subject=20  # number of sample to use for evaluation
)

abstract_algebra: 0.3500
anatomy: 0.7000
astronomy: 1.0000
business_ethics: 0.9000
clinical_knowledge: 0.7500
college_biology: 0.8000
college_chemistry: 0.4500
college_computer_science: 0.6000
college_mathematics: 0.2500
college_medicine: 0.9000
college_physics: 0.8000
computer_security: 0.8000
conceptual_physics: 0.9000
econometrics: 0.6500
electrical_engineering: 0.8500
elementary_mathematics: 0.7000
formal_logic: 0.7000
global_facts: 0.4500
high_school_biology: 0.8500
high_school_chemistry: 0.7000
high_school_computer_science: 1.0000
high_school_european_history: 0.8000
high_school_geography: 0.8500
high_school_government_and_politics: 0.9500
high_school_macroeconomics: 0.7500
high_school_mathematics: 0.3500
high_school_microeconomics: 0.6500
high_school_physics: 0.5500
high_school_psychology: 0.9500
high_school_statistics: 0.8000
high_school_us_history: 0.8500
high_school_world_history: 0.9000
human_aging: 0.6500
human_sexuality: 0.8000
international_law: 0.8500
jurisprudence: 0.75

In [ ]:
dict_for_save = {'results': results, 'scores': score}

In [ ]:
print(results)
print(score)

{'abstract_algebra': 0.4, 'anatomy': 0.7, 'astronomy': 1.0, 'business_ethics': 0.9, 'clinical_knowledge': 0.75, 'college_biology': 0.8, 'college_chemistry': 0.45, 'college_computer_science': 0.6, 'college_mathematics': 0.25, 'college_medicine': 0.9, 'college_physics': 0.8, 'computer_security': 0.8, 'conceptual_physics': 0.85, 'econometrics': 0.65, 'electrical_engineering': 0.85, 'elementary_mathematics': 0.7, 'formal_logic': 0.7, 'global_facts': 0.45, 'high_school_biology': 0.85, 'high_school_chemistry': 0.75, 'high_school_computer_science': 1.0, 'high_school_european_history': 0.8, 'high_school_geography': 0.85, 'high_school_government_and_politics': 0.95, 'high_school_macroeconomics': 0.75, 'high_school_mathematics': 0.35, 'high_school_microeconomics': 0.65, 'high_school_physics': 0.55, 'high_school_psychology': 0.95, 'high_school_statistics': 0.75, 'high_school_us_history': 0.85, 'high_school_world_history': 0.9, 'human_aging': 0.65, 'human_sexuality': 0.8, 'international_law': 0.85